In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.cloud import bigquery

# Localizamos la raíz del proyecto.
# El notebook está dentro de:
# parte_2_modelo_bigquery/notebooks/
# Por eso subimos dos niveles.
PROJECT_ROOT = Path.cwd().parents[1]

# Cargamos el archivo .env situado en la raíz del proyecto.
load_dotenv(PROJECT_ROOT / ".env", override=True)

# Leemos la configuración del proyecto.
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

# Construimos la ruta absoluta al archivo de credenciales.
CREDENTIALS_PATH = PROJECT_ROOT / os.getenv(
    "GOOGLE_APPLICATION_CREDENTIALS"
)

# Convertimos la ruta a string y la establecemos para Google Cloud.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(CREDENTIALS_PATH)

print("Proyecto:", PROJECT_ID)
print("Dataset:", DATASET_ID)
print("Credenciales:", CREDENTIALS_PATH)
print("¿Existe el archivo?:", CREDENTIALS_PATH.exists())

Proyecto: tc-sql-miguel
Dataset: electromarket
Credenciales: c:\Users\mgpir\Desktop\RepoPracticaObligatoria\bootcamp_AI_Engineering_05_26\tc-sql-lopezmiguel\credentials\service-account.json
¿Existe el archivo?: True


In [3]:
# Creamos el cliente de BigQuery utilizando
# el proyecto y las credenciales configuradas anteriormente.

client = bigquery.Client(project=PROJECT_ID)

print("Conexión con BigQuery correcta.")

Conexión con BigQuery correcta.


In [4]:
# Construimos la referencia al dataset utilizando
# el ID del proyecto y el nombre del dataset.

dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"

# Creamos la configuración del dataset.
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "EU"

# Creamos el dataset si no existe.
# Si ya existe, exists_ok=True evita que se produzca un error.

dataset = client.create_dataset(dataset, exists_ok=True)

print(f"Dataset creado/verificado: {dataset_ref}")

Dataset creado/verificado: tc-sql-miguel.electromarket


In [5]:
# Definimos el esquema de la tabla customers.
customers_schema = [
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("first_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("last_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("phone", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("city", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("acquisition_channel", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("registration_date", "DATE", mode="REQUIRED"),
]

In [6]:
# Definimos el esquema de la tabla categories.
categories_schema = [
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
]

In [7]:
# Definimos el esquema de la tabla products.
products_schema = [
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("cost", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("stock", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("is_active", "BOOL", mode="REQUIRED"),
]

In [8]:
# Definimos el esquema de la tabla orders.
orders_schema = [
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("shipping_address", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("shipping_city", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("shipping_country", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("order_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("shipped_date", "DATE", mode="NULLABLE"),
    bigquery.SchemaField("delivered_date", "DATE", mode="NULLABLE"),
]

In [12]:
# Definimos el esquema de la tabla order_items.
order_items_schema = [
    bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("quantity", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("unit_price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("discount", "NUMERIC", mode="REQUIRED"),
]

In [10]:
# Definimos el esquema de la tabla payments.
payments_schema = [
    bigquery.SchemaField("payment_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("payment_method", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("payment_status", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("amount", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("payment_date", "DATE", mode="REQUIRED"),
]

In [11]:
# Definimos el esquema de la tabla reviews.
reviews_schema = [
    bigquery.SchemaField("review_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("rating", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("comment", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("review_date", "DATE", mode="REQUIRED"),
]

In [13]:
# Creamos una función reutilizable para crear tablas en BigQuery.
def crear_tabla(nombre_tabla, esquema):
    tabla_ref = f"{PROJECT_ID}.{DATASET_ID}.{nombre_tabla}"

    tabla = bigquery.Table(tabla_ref, schema=esquema)

    # Creamos la tabla o la dejamos intacta si ya existe.
    tabla = client.create_table(tabla, exists_ok=True)

    print(f"Tabla creada/verificada: {tabla_ref}")

In [14]:
# Creamos las tablas independientes.
crear_tabla("customers", customers_schema)
crear_tabla("categories", categories_schema)

Tabla creada/verificada: tc-sql-miguel.electromarket.customers
Tabla creada/verificada: tc-sql-miguel.electromarket.categories


In [15]:
# Creamos products, relacionada con categories.
crear_tabla("products", products_schema)

Tabla creada/verificada: tc-sql-miguel.electromarket.products


In [16]:
# Creamos orders, relacionada con customers.
crear_tabla("orders", orders_schema)

Tabla creada/verificada: tc-sql-miguel.electromarket.orders


In [17]:
# Creamos order_items, tabla intermedia entre orders y products.
crear_tabla("order_items", order_items_schema)

Tabla creada/verificada: tc-sql-miguel.electromarket.order_items


In [18]:
# Creamos payments, relacionada con orders.
crear_tabla("payments", payments_schema)

Tabla creada/verificada: tc-sql-miguel.electromarket.payments


In [19]:
# Creamos reviews, relacionada con order_items.
crear_tabla("reviews", reviews_schema)

Tabla creada/verificada: tc-sql-miguel.electromarket.reviews


In [20]:
# Consultamos las tablas existentes en nuestro dataset.
tablas = list(client.list_tables(dataset_ref))

print("Tablas existentes:")

for tabla in tablas:
    print(f"- {tabla.table_id}")

Tablas existentes:
- categories
- customers
- order_items
- orders
- payments
- products
- reviews


In [21]:
# Comprobamos que las 7 tablas obligatorias existen.
tablas_esperadas = {
    "customers",
    "categories",
    "products",
    "orders",
    "order_items",
    "payments",
    "reviews",
}

tablas_creadas = {tabla.table_id for tabla in client.list_tables(dataset_ref)}

faltantes = tablas_esperadas - tablas_creadas

if not faltantes:
    print("Todas las tablas obligatorias están creadas correctamente.")
else:
    print("Faltan las siguientes tablas:")
    for tabla in faltantes:
        print(f"- {tabla}")

Todas las tablas obligatorias están creadas correctamente.


In [22]:
# Mostramos las columnas y tipos de cada tabla.
for nombre_tabla in sorted(tablas_esperadas):
    tabla_ref = f"{PROJECT_ID}.{DATASET_ID}.{nombre_tabla}"
    tabla = client.get_table(tabla_ref)

    print(f"\n--- {nombre_tabla} ---")

    for campo in tabla.schema:
        print(f"{campo.name}: {campo.field_type} ({campo.mode})")


--- categories ---
category_id: INTEGER (REQUIRED)
name: STRING (REQUIRED)
description: STRING (NULLABLE)

--- customers ---
customer_id: INTEGER (REQUIRED)
first_name: STRING (REQUIRED)
last_name: STRING (REQUIRED)
email: STRING (REQUIRED)
phone: STRING (NULLABLE)
country: STRING (REQUIRED)
city: STRING (REQUIRED)
acquisition_channel: STRING (REQUIRED)
registration_date: DATE (REQUIRED)

--- order_items ---
order_item_id: INTEGER (REQUIRED)
order_id: INTEGER (REQUIRED)
product_id: INTEGER (REQUIRED)
quantity: INTEGER (REQUIRED)
unit_price: NUMERIC (REQUIRED)
discount: NUMERIC (REQUIRED)

--- orders ---
order_id: INTEGER (REQUIRED)
customer_id: INTEGER (REQUIRED)
status: STRING (REQUIRED)
shipping_address: STRING (REQUIRED)
shipping_city: STRING (REQUIRED)
shipping_country: STRING (REQUIRED)
order_date: DATE (REQUIRED)
shipped_date: DATE (NULLABLE)
delivered_date: DATE (NULLABLE)

--- payments ---
payment_id: INTEGER (REQUIRED)
order_id: INTEGER (REQUIRED)
payment_method: STRING (REQU